# Лабораторная работа

**Сверточные нейронные сети**



## Загрузка данных

В данной работе будет использоваться учебный датасет с изображениями персонажей из Симпсонов. Загрузите файлы и распакуйте, вo вкладке Files должна появиться папка `data`, а в ней папки `train` и `test`.

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# 2. Пути к архивам на Google Диске
zip_paths = [
    "/content/drive/My Drive/rgb-test.zip",
    "/content/drive/My Drive/rgb-train.zip"
]

In [12]:
import os
import time
import shutil

# Папка для распаковки
extract_path = "data"
os.makedirs(extract_path, exist_ok=True)  # Создаём, если нет

start_time = time.time()

for zip_file in zip_paths:
    if os.path.exists(zip_file):  # Проверяем, существует ли файл
        try:
            shutil.unpack_archive(zip_file, extract_path)
            print(f"Успешно распаковано: {zip_file}")
        except Exception as e:
            print(f"Ошибка при распаковке {zip_file}: {e}")
    else:
        print(f"Файл не найден: {zip_file}")

print(f"Архивы обработаны за {time.time() - start_time:.2f} сек")

Успешно распаковано: /content/drive/My Drive/rgb-test.zip
Успешно распаковано: /content/drive/My Drive/rgb-train.zip
Архивы обработаны за 4.05 сек


In [13]:
print("Train data:", os.listdir("data/train")[:5])
print("Test data:", os.listdir("data/test")[:5])

Train data: ['bart_simpson', 'charles_montgomery_burns', 'lisa_simpson', 'marge_simpson', 'krusty_the_clown']
Test data: ['bart_simpson', 'charles_montgomery_burns', 'lisa_simpson', 'marge_simpson', 'krusty_the_clown']


## Создание объекта Dataset

Так как изображения в датасете организованы по папкам, где имя папки является ярлыком для данных, то мы можем воспользоваться базовым классом `ImageFolder`.

Одним из параметров является `transform`, для которого необходимо скомпоновать преобразования для наших изображений. В pytorch для преобразований есть два набора функций V1 и V2 и рекомендуется использовать V2.

Для компоновки функции из модуля v2 используйте `Compose`.Параметры:
* ToImage() - преобразование в `Image` (подкласс torch.Tensor)
* RandomVerticalFlip() - случайное отзеркаливание
* ToDtype(torch.float32, scale=True) - преобразование из int во float
* Normalize() - нормализация изображений по полученным средним и стандартным отклонениям.

По желанию:
* RandomRotation() - поворот на случайный угол в указанном диапазоне
* Можете попробовать и другие варианты преобразований. [Документация API V2](https://pytorch.org/vision/stable/transforms.html#v2-api-reference-recommended)

 При обучении используете весь набор обязательных преобразований, при проверке требуется только преобразовать изображение к тензору с плавающей точкой и провести нормализацию.

### Получение статистик для нормализации

In [14]:
import torch
from torchvision.transforms import v2

transforms_stats = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

In [15]:
from torchvision.datasets import ImageFolder

stats_dataset = ImageFolder(root="./data/train", transform=transforms_stats)

In [16]:
imgs = [item[0] for item in stats_dataset]
imgs = torch.stack(imgs, dim=0).numpy()

mean_r = imgs[:,0,:,:].mean()
mean_g = imgs[:,1,:,:].mean()
mean_b = imgs[:,2,:,:].mean()
print(f"Means R, G, B: {mean_r,mean_g,mean_b}")

std_r = imgs[:,0,:,:].std()
std_g = imgs[:,1,:,:].std()
std_b = imgs[:,2,:,:].std()
print(f"Std R, G, B: {std_r,std_g,std_b}")

Means R, G, B: (0.465204, 0.40914717, 0.35447764)
Std R, G, B: (0.2492806, 0.22941421, 0.2450912)



Используйте выведенные выше значения средних и стандартных отклонений в качестве аргументов функции `Normalize`.

In [17]:
transforms_train = v2.Compose([
    # Ваш код
    v2.ToImage(),
    v2.RandomVerticalFlip(p=0.5),
    v2.RandomRotation(15),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.465204, 0.40914717, 0.35447764), std=(0.2492806, 0.22941421, 0.2450912))
])

transforms_test = v2.Compose([
    # Ваш код
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=(0.465204, 0.40914717, 0.35447764), std=(0.2492806, 0.22941421, 0.2450912))
])

Теперь, когда есть необходимые `transforms` можно создать ImageFolder, указав в `root` путь до выборки и `transforms` в `transform`.

In [18]:
train_dataset = ImageFolder(root="./data/train", transform=transforms_train) #  Ваш код

# по аналогии создайте датасет для проверочной части

#
#  Ваш код
#
test_dataset = ImageFolder(root="./data/test", transform=transforms_test)

In [19]:
# Выведите информацию о количестве изображений в обеих выборках и количество классов.
# Воспользуйтесь атрибутом classes для получения количества классов.
#
# Ваш код
#
print(f"Количество изображений в train_dataset: {len(train_dataset)}")
print(f"Количество изображений в test_dataset: {len(test_dataset)}")
print(f"Количество классов: {len(train_dataset.classes)}")

Количество изображений в train_dataset: 8000
Количество изображений в test_dataset: 2000
Количество классов: 10


## Создание DataLoader

Далее необходимо подготовить три загрузчика данных:

1. Обучающий
2. Проверочный
3. Тестовый

Тестовый загрузчик делается из тестового Dataset, а обучающий и проверочный необходимо создать, используя [SubsetRandomSampler](https://pytorch.org/docs/stable/data.html#torch.utils.data.SubsetRandomSampler), для его работы требуется массив индексов, по которым в дальнейшем загрузчик будет в случайном порядке брать изображения и лейблы.

In [28]:
import numpy as np
from torch.utils.data.sampler import SubsetRandomSampler

val_size = 0.2

train_samples = len(train_dataset)
train_indices = list(range(train_samples))

split_value = int(np.floor(val_size * train_samples))
np.random.shuffle(train_indices)

train_idx, val_idx = train_indices[split_value:], train_indices[:split_value]

train_sampler = SubsetRandomSampler(train_idx)
val_sampler = SubsetRandomSampler(val_idx)

In [20]:
from torch.utils.data import DataLoader

BATCH_SIZE = 32 # Ваш код

## Создание модели


Сохдайте модель сети.
Сверточные нейронные сети состоят из двух частей:
1. Слои свертки(функции свертки, активации, субдискретизации)
2. Полносвязные слои (MLP)

Слои можно объединить с помощью `nn.Sequential()`. А класс модели должен наследоваться от `nn.Module`.

`def forward()` определяет прямой ход и должна возвращать итоговый результат работы модели - в данном случае логиты.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

class ConvNet(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.layer2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        # Правильный размер входа для fc1
        self.fc1 = nn.Linear(64 * 8 * 8, 512)  # Размер 64*8*8
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = x.view(x.size(0), -1)  # Преобразуем в вектор
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        out = self.fc4(x)
        return out

# Преобразования данных
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# Загрузка данных CIFAR-10
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Функция потерь и оптимизатор
model = ConvNet(num_classes=10)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Тренировка модели
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for i, (images, labels) in enumerate(train_loader):
        optimizer.zero_grad()

        # Прямой ход
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Обратный ход
        loss.backward()
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')

    # Валидация
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in val_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f'Accuracy: {accuracy}%')

100%|██████████| 170M/170M [00:03<00:00, 47.1MB/s]


Extracting ./data/cifar-10-python.tar.gz to ./data
Files already downloaded and verified


## Создание объекта модели, функции потерь и оптимизатора

В качестве функции потерь будет использована перекрестная энтропия.

В качестве оптимизатора можете взять стохастический градиентный спуск или Adam.

In [43]:
import torch

# Задаем количество классов, эпох, скорость обучения
num_classes = 10  # Ваше число
num_epochs = 10  # Пробуйте разные значения
learning_rate = 0.005  # Пробуйте разные значения

# Создаем объект модели
model = ConvNet(num_classes)

# Функция потерь - перекрестная энтропия
loss_fn = nn.CrossEntropyLoss()

# Оптимизатор - стохастический градиентный спуск
optimizer = torch.optim.SGD(
    # Ваш код
    model.parameters(),  # Параметры модели, которые будем оптимизировать
    lr=learning_rate,
    momentum=0.9  # Моментум, чтобы ускорить обучение
)

## Цикл обучения

Попробуйте разные гиперпараметры для вашей модели. Попробуйте улучшить результат от первой попытки.

Можно вывести графики потерь и точности от эпохи.

In [44]:
# Определяем DataLoader для обучающей выборки
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=4
)

# Определяем DataLoader для проверочной выборки
val_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=val_sampler,
    num_workers=4
)

# Определяем DataLoader для тестовой выборки
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    num_workers=4
)

In [51]:
# Проверка размеров после свертки и подвыборки
def get_conv_output_size(model, input_size=(3, 32, 32)):
    with torch.no_grad():
        x = torch.randn(1, *input_size)  # Создаем случайный тензор с размером (1, 3, 32, 32)
        x = model.layer1(x)  # Пропускаем через первый сверточный слой
        x = model.layer2(x)  # Пропускаем через второй сверточный слой
        return x.shape

# Получаем размер выхода после свертки
conv_output_size = get_conv_output_size(model)
print(f"Размер выхода после свертки и подвыборки: {conv_output_size}")

Размер выхода после свертки и подвыборки: torch.Size([1, 64, 8, 8])


In [49]:
%%time

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_loader):
        model.train()

        # Forward pass
        # Прямой ход:
        outputs = model(images)  # Расчет вывода модели
        loss = loss_fn(outputs, labels)  # Расчет потерь

        # Обнуление градиентов
        optimizer.zero_grad()

        # Обратный ход
        loss.backward()

        # Шаг оптимизатора
        optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')

    # Валидация
    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in val_loader:  # Используем val_loader для валидации
            outputs = model(images)  # Расчет вывода модели

            # Получаем индексы класса с максимальной вероятностью
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)  # Количество данных в этом батче
            correct += (predicted == labels).sum().item()  # Сумма совпадений

        accuracy = 100 * correct / total
        print(f'Точность (accuracy) {accuracy}%')

RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x3136 and 4096x512)

In [47]:
for images, labels in train_loader:
    print(images.shape)  # Должно быть [batch_size, 3, 32, 32]
    break

torch.Size([32, 3, 28, 28])


## Итоговая оценка

Часть кода, которую вы реализовали для оценки модели на валидационной выборке, можно использовать для финальной проверки, указав нужный loader.

In [ ]:
with torch.no_grad():
    model.eval()
    correct = 0
    total = 0
    for images, labels in ___:
      #
      # Ваш код
      #

    print(f'Точность (accuracy) {100 * correct / total}')

#  Задание 1.
1. Загрузите датасет Cifar-10
https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html
2.	Создайте 3 архитектуры сверточной нейронной сети для распознавания данных из набора Cifar-10. Обучите сеть и выведите результаты на тренировочной и валидационной выборках.
4.	Проверьте работу сети на тестовой выборке.
5.	Загрузите файлы с картинками из интернета или создайте свои фотографии.
6.	Проверьте работу сети на собственных картинках.

#  Задание 2.
1. Загрузите датасет Cifar-100
https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR100.html
2.	Создайте 3 архитектуры сверточной нейронной сети для распознавания данных из набора Cifar-10. Обучите сеть и выведите результаты на тренировочной и валидационной выборках.
4.	Проверьте работу сети на тестовой выборке.
5.	Загрузите файлы с картинками из интернета или создайте свои фотографии.
6.	Проверьте работу сети на собственных картинках.